In [1]:
!pip install papermill
!pip install flake8
!pip install pytest
!pip install python-dotenv
!pip install joblib
!pip install ipykernel

In [2]:
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import papermill as pm
import flake8
import pytest
import dotenv

In [3]:
train_processed = pd.read_csv('train_processed.csv')

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, f1_score, precision_score, recall_score

X = train_processed.drop(columns=['Class'])
y = train_processed['Class']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f"Baseline сплит: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

Baseline сплит: train (48292, 8), val (16098, 8), test (16098, 8)


In [5]:
lr_baseline = LogisticRegression(random_state=42, class_weight='balanced')
lr_baseline.fit(X_train_scaled, y_train)

y_val_proba = lr_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_val = log_loss(y_val, y_val_proba)
roc_auc_val = roc_auc_score(y_val, y_val_proba)

from sklearn.metrics import precision_recall_curve
precision, recall, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores[:-1])]
y_val_pred = (y_val_proba >= best_threshold).astype(int)

f1_val = f1_score(y_val, y_val_pred)
precision_val = precision_score(y_val, y_val_pred)
recall_val = recall_score(y_val, y_val_pred)

print("Logistic Regression")
print(f"LogLoss: {logloss_val:.4f}")
print(f"ROC-AUC: {roc_auc_val:.4f}")
print(f"F1 (порог={best_threshold:.3f}): {f1_val:.4f}")
print(f"Precision: {precision_val:.4f}")
print(f"Recall: {recall_val:.4f}")

Logistic Regression
LogLoss: 0.4067
ROC-AUC: 0.8108
F1 (порог=0.968): 0.1022
Precision: 0.0619
Recall: 0.2917


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_recall_curve

knn_baseline = KNeighborsClassifier(n_neighbors=5)
knn_baseline.fit(X_train_scaled, y_train)
y_val_proba_knn = knn_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_knn = log_loss(y_val, y_val_proba_knn)
roc_auc_knn = roc_auc_score(y_val, y_val_proba_knn)

precision_knn, recall_knn, thresholds_knn = precision_recall_curve(y_val, y_val_proba_knn)
f1_scores_knn = 2 * precision_knn * recall_knn / (precision_knn + recall_knn + 1e-9)
best_threshold_knn = thresholds_knn[np.argmax(f1_scores_knn[:-1])]
y_val_pred_knn = (y_val_proba_knn >= best_threshold_knn).astype(int)

f1_knn = f1_score(y_val, y_val_pred_knn)
precision_knn_val = precision_score(y_val, y_val_pred_knn)
recall_knn_val = recall_score(y_val, y_val_pred_knn)

print("KNN (k=5)")
print(f"LogLoss: {logloss_knn:.4f}")
print(f"ROC-AUC: {roc_auc_knn:.4f}")
print(f"F1 (порог={best_threshold_knn:.3f}): {f1_knn:.4f}")
print(f"Precision: {precision_knn_val:.4f}")
print(f"Recall: {recall_knn_val:.4f}")

KNN (k=5)
LogLoss: 0.0487
ROC-AUC: 0.5595
F1 (порог=0.200): 0.0484
Precision: 0.0300
Recall: 0.1250


In [7]:
from sklearn.tree import DecisionTreeClassifier

dt_baseline = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_baseline.fit(X_train_scaled, y_train)
y_val_proba_dt = dt_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_dt = log_loss(y_val, y_val_proba_dt)
roc_auc_dt = roc_auc_score(y_val, y_val_proba_dt)

precision_dt, recall_dt, thresholds_dt = precision_recall_curve(y_val, y_val_proba_dt)
f1_scores_dt = 2 * precision_dt * recall_dt / (precision_dt + recall_dt + 1e-9)
best_threshold_dt = thresholds_dt[np.argmax(f1_scores_dt[:-1])]
y_val_pred_dt = (y_val_proba_dt >= best_threshold_dt).astype(int)

f1_dt = f1_score(y_val, y_val_pred_dt)
precision_dt_val = precision_score(y_val, y_val_pred_dt)
recall_dt_val = recall_score(y_val, y_val_pred_dt)

print("Decision Tree")
print(f"LogLoss: {logloss_dt:.4f}")
print(f"ROC-AUC: {roc_auc_dt:.4f}")
print(f"F1 (порог={best_threshold_dt:.3f}): {f1_dt:.4f}")
print(f"Precision: {precision_dt_val:.4f}")
print(f"Recall: {recall_dt_val:.4f}")

Decision Tree
LogLoss: 0.0873
ROC-AUC: 0.4995
F1 (порог=0.000): 0.0030
Precision: 0.0015
Recall: 1.0000


In [8]:
from sklearn.naive_bayes import GaussianNB

nb_baseline = GaussianNB()
nb_baseline.fit(X_train_scaled, y_train)
y_val_proba_nb = nb_baseline.predict_proba(X_val_scaled)[:, 1]

logloss_nb = log_loss(y_val, y_val_proba_nb)
roc_auc_nb = roc_auc_score(y_val, y_val_proba_nb)

prec_nb, rec_nb, thresh_nb = precision_recall_curve(y_val, y_val_proba_nb)
f1_nb_curve = 2 * prec_nb * rec_nb / (prec_nb + rec_nb + 1e-9)
best_th_nb = thresh_nb[np.argmax(f1_nb_curve[:-1])]
y_val_pred_nb = (y_val_proba_nb >= best_th_nb).astype(int)

f1_nb = f1_score(y_val, y_val_pred_nb)
precision_nb = precision_score(y_val, y_val_pred_nb)
recall_nb = recall_score(y_val, y_val_pred_nb)

print("Gaussian Naive Bayes")
print(f"LogLoss: {logloss_nb:.4f}")
print(f"ROC-AUC: {roc_auc_nb:.4f}")
print(f"F1 (порог={best_th_nb:.3f}): {f1_nb:.4f}")
print(f"Precision: {precision_nb:.4f}")
print(f"Recall: {recall_nb:.4f}\n")

Gaussian Naive Bayes
LogLoss: 0.0333
ROC-AUC: 0.7809
F1 (порог=0.632): 0.0526
Precision: 0.0333
Recall: 0.1250



In [9]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_baseline = LinearSVC(random_state=42, class_weight='balanced', max_iter=2000)
svm_calibrated = CalibratedClassifierCV(svm_baseline, method='sigmoid', cv=3)
svm_calibrated.fit(X_train_scaled, y_train)
y_val_proba_svm = svm_calibrated.predict_proba(X_val_scaled)[:, 1]

logloss_svm = log_loss(y_val, y_val_proba_svm)
roc_auc_svm = roc_auc_score(y_val, y_val_proba_svm)

prec_svm, rec_svm, thresh_svm = precision_recall_curve(y_val, y_val_proba_svm)
f1_svm_curve = 2 * prec_svm * rec_svm / (prec_svm + rec_svm + 1e-9)
best_th_svm = thresh_svm[np.argmax(f1_svm_curve[:-1])]
y_val_pred_svm = (y_val_proba_svm >= best_th_svm).astype(int)

f1_svm = f1_score(y_val, y_val_pred_svm)
precision_svm = precision_score(y_val, y_val_pred_svm)
recall_svm = recall_score(y_val, y_val_pred_svm)

print("Linear SVM (calibrated)")
print(f"LogLoss: {logloss_svm:.4f}")
print(f"ROC-AUC: {roc_auc_svm:.4f}")
print(f"F1 (порог={best_th_svm:.3f}): {f1_svm:.4f}")
print(f"Precision: {precision_svm:.4f}")
print(f"Recall: {recall_svm:.4f}")

Linear SVM (calibrated)
LogLoss: 0.0099
ROC-AUC: 0.8061
F1 (порог=0.025): 0.1000
Precision: 0.0625
Recall: 0.2500


In [10]:
results_baseline = {
    'Model': ['Logistic Regression', 'KNN (k=5)', 'Decision Tree',
              'Gaussian NB', 'Linear SVM'],
    'LogLoss': [logloss_val, logloss_knn, logloss_dt,
                logloss_nb, logloss_svm],
    'ROC-AUC': [roc_auc_val, roc_auc_knn, roc_auc_dt,
                roc_auc_nb, roc_auc_svm],
    'F1': [f1_val, f1_knn, f1_dt,
           f1_nb, f1_svm],
    'Precision': [precision_val, precision_knn_val, precision_dt_val,
                  precision_nb, precision_svm],
    'Recall': [recall_val, recall_knn_val, recall_dt_val,
               recall_nb, recall_svm]
}

df_results = pd.DataFrame(results_baseline)
df_results = df_results.sort_values('LogLoss')

print("Сравнение baseline моделей")
print(df_results.to_string(index=False))

Сравнение baseline моделей
              Model  LogLoss  ROC-AUC       F1  Precision   Recall
         Linear SVM 0.009925 0.806066 0.100000   0.062500 0.250000
        Gaussian NB 0.033321 0.780907 0.052632   0.033333 0.125000
          KNN (k=5) 0.048682 0.559479 0.048387   0.030000 0.125000
      Decision Tree 0.087322 0.499533 0.002977   0.001491 1.000000
Logistic Regression 0.406711 0.810761 0.102190   0.061947 0.291667


Среди базовых моделей наилучший результат по главной метрике LogLoss показал Linear SVM – это более чем в три раза лучше, чем у второго места Gaussian Naive Bayes. Такой низкий LogLoss свидетельствует об отличной калибровке вероятностей и крайне редких уверенных ошибочных предсказаниях, что критически важно для соревнования Kaggle. При этом Linear SVM также продемонстрировал высокий ROC‑AUC, уступая только логистической регрессии. Gaussian Naive Bayes показал умеренные результаты, но уступает как Linear SVM, так и логистической регрессии по разделяющей способности. Логистическая регрессия, несмотря на самый высокий ROC‑AUC, имеет катастрофически высокий LogLoss, что делает её непригодной для практического использования – она плохо калибрует вероятности. KNN и решающее дерево показали вырожденные или переобученные результаты: KNN даёт аномально низкий LogLoss за счёт предсказаний, близких к нулю, но теряет пульсары, а дерево достигло анамально высокого Recall ценой почти случайного ROC‑AUC. Таким образом, среди всех baseline-моделей Linear SVM является лучшей по LogLoss и может рассматриваться как сильный базовый алгоритм.

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train_scaled, y_train)
y_val_proba_rf = rf.predict_proba(X_val_scaled)[:, 1]

logloss_rf = log_loss(y_val, y_val_proba_rf)
roc_auc_rf = roc_auc_score(y_val, y_val_proba_rf)

prec_rf, rec_rf, thresh_rf = precision_recall_curve(y_val, y_val_proba_rf)
f1_rf = 2 * prec_rf * rec_rf / (prec_rf + rec_rf + 1e-9)
best_th_rf = thresh_rf[np.argmax(f1_rf[:-1])]
y_val_pred_rf = (y_val_proba_rf >= best_th_rf).astype(int)

f1_score_rf = f1_score(y_val, y_val_pred_rf)
precision_rf = precision_score(y_val, y_val_pred_rf)
recall_rf = recall_score(y_val, y_val_pred_rf)

print("Random Forest")
print(f"LogLoss: {logloss_rf:.4f}")
print(f"ROC-AUC: {roc_auc_rf:.4f}")
print(f"F1 (порог={best_th_rf:.3f}): {f1_score_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall: {recall_rf:.4f}\n")

Random Forest
LogLoss: 0.0565
ROC-AUC: 0.7603
F1 (порог=0.562): 0.0377
Precision: 0.0345
Recall: 0.0417



In [12]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),  # балансировка
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_train_scaled, y_train)
y_val_proba_xgb = xgb.predict_proba(X_val_scaled)[:, 1]

logloss_xgb = log_loss(y_val, y_val_proba_xgb)
roc_auc_xgb = roc_auc_score(y_val, y_val_proba_xgb)

prec_xgb, rec_xgb, thresh_xgb = precision_recall_curve(y_val, y_val_proba_xgb)
f1_xgb = 2 * prec_xgb * rec_xgb / (prec_xgb + rec_xgb + 1e-9)
best_th_xgb = thresh_xgb[np.argmax(f1_xgb[:-1])]
y_val_pred_xgb = (y_val_proba_xgb >= best_th_xgb).astype(int)

f1_score_xgb = f1_score(y_val, y_val_pred_xgb)
precision_xgb = precision_score(y_val, y_val_pred_xgb)
recall_xgb = recall_score(y_val, y_val_pred_xgb)

print("XGBoost")
print(f"LogLoss: {logloss_xgb:.4f}")
print(f"ROC-AUC: {roc_auc_xgb:.4f}")
print(f"F1 (порог={best_th_xgb:.3f}): {f1_score_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall: {recall_xgb:.4f}\n")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [22:23:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost
LogLoss: 0.0302
ROC-AUC: 0.6866
F1 (порог=0.777): 0.0417
Precision: 0.0417
Recall: 0.0417



In [13]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)
lgbm.fit(X_train_scaled, y_train)
y_val_proba_lgb = lgbm.predict_proba(X_val_scaled)[:, 1]

logloss_lgb = log_loss(y_val, y_val_proba_lgb)
roc_auc_lgb = roc_auc_score(y_val, y_val_proba_lgb)

prec_lgb, rec_lgb, thresh_lgb = precision_recall_curve(y_val, y_val_proba_lgb)
f1_lgb = 2 * prec_lgb * rec_lgb / (prec_lgb + rec_lgb + 1e-9)
best_th_lgb = thresh_lgb[np.argmax(f1_lgb[:-1])]
y_val_pred_lgb = (y_val_proba_lgb >= best_th_lgb).astype(int)

f1_score_lgb = f1_score(y_val, y_val_pred_lgb)
precision_lgb = precision_score(y_val, y_val_pred_lgb)
recall_lgb = recall_score(y_val, y_val_pred_lgb)

print("LightGBM")
print(f"LogLoss: {logloss_lgb:.4f}")
print(f"ROC-AUC: {roc_auc_lgb:.4f}")
print(f"F1 (порог={best_th_lgb:.3f}): {f1_score_lgb:.4f}")
print(f"Precision: {precision_lgb:.4f}")
print(f"Recall: {recall_lgb:.4f}")

LightGBM
LogLoss: 0.0324
ROC-AUC: 0.6517
F1 (порог=0.595): 0.0227
Precision: 0.0156
Recall: 0.0417


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [14]:
results = {
    'Model': ['Logistic Regression', 'KNN (k=5)', 'Decision Tree',
              'Gaussian NB', 'Linear SVM',
              'Random Forest', 'XGBoost', 'LightGBM'],
    'LogLoss': [logloss_val, logloss_knn, logloss_dt,
                logloss_nb, logloss_svm,
                logloss_rf, logloss_xgb, logloss_lgb],
    'ROC-AUC': [roc_auc_val, roc_auc_knn, roc_auc_dt,
                roc_auc_nb, roc_auc_svm,
                roc_auc_rf, roc_auc_xgb, roc_auc_lgb],
    'F1': [f1_val, f1_knn, f1_dt,
           f1_nb, f1_svm,
           f1_score_rf, f1_score_xgb, f1_score_lgb],
    'Precision': [precision_val, precision_knn_val, precision_dt_val,
                  precision_nb, precision_svm,
                  precision_rf, precision_xgb, precision_lgb],
    'Recall': [recall_val, recall_knn_val, recall_dt_val,
               recall_nb, recall_svm,
               recall_rf, recall_xgb, recall_lgb]
}

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('LogLoss')

print("Сравнение моделей")
print(df_results.to_string(index=False))

Сравнение моделей
              Model  LogLoss  ROC-AUC       F1  Precision   Recall
         Linear SVM 0.009925 0.806066 0.100000   0.062500 0.250000
            XGBoost 0.030202 0.686632 0.041667   0.041667 0.041667
           LightGBM 0.032365 0.651747 0.022727   0.015625 0.041667
        Gaussian NB 0.033321 0.780907 0.052632   0.033333 0.125000
          KNN (k=5) 0.048682 0.559479 0.048387   0.030000 0.125000
      Random Forest 0.056485 0.760259 0.037736   0.034483 0.041667
      Decision Tree 0.087322 0.499533 0.002977   0.001491 1.000000
Logistic Regression 0.406711 0.810761 0.102190   0.061947 0.291667


Среди всех обученных моделей наилучший результат по главной метрике LogLoss показал Linear SVM, что значительно превосходит остальные алгоритмы. Ансамблевые методы XGBoost и LightGBM уступили линейной модели более чем в три раза, несмотря на свою сложность. Gaussian Naive Bayes и KNN также оказались хуже, а Random Forest — тем более. Логистическая регрессия, несмотря на самый высокий ROC‑AUC, провалилась по LogLoss из‑за плохой калибровки вероятностей, а решающее дерево и KNN показали вырожденные или переобученные результаты. Таким образом, Linear SVM является безусловным лидером по главной метрике, одновременно демонстрируя высокий ROC‑AUC и сбалансированные Precision/Recall.

Финальной моделью для решения задачи бинарной классификации пульсаров выбирается Linear SVM на основе достигнутого минимального значения LogLoss = 0,009925 — это главный критерий успеха в соревновании Kaggle, отражающий наилучшую калибровку вероятностей. Данный показатель более чем в три раза лучше, чем у ближайшего конкурента (Gaussian NB), и в разы превосходит все ансамблевые методы. Линейный SVM также обеспечивает высокий ROC‑AUC (0,806), что подтверждает его способность разделять классы, и приемлемые значения Precision (0,0625) и Recall (0,25) с учётом крайнего дисбаланса (пульсары ≈0,15% выборки). Следовательно, Linear SVM является оптимальным выбором для дальнейшего тестирования на отложенной выборке и возможного развёртывания.

Проведём подбор гиперпараметров для трёх лучших моделей, показавших наименьший LogLoss на предыдущем этапе: Linear SVM, XGBoost и LightGBM. Для каждой модели будем перебирать ключевые параметры: для SVM – C, loss, tol; для XGBoost и LightGBM – n_estimators, max_depth, learning_rate, subsample, colsample_bytree. Цель – настроить параметры для минимизации LogLoss на валидации, улучшив калибровку вероятностей и избежав переобучения.

In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_base = LinearSVC(random_state=42, class_weight='balanced', max_iter=10000)

param_grid_svm = {
    'C': [0.01, 0.1, 1, 10, 100],
    'loss': ['hinge', 'squared_hinge'],
    'tol': [1e-4, 1e-3]
}

grid_svm = GridSearchCV(svm_base, param_grid_svm, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_svm.fit(X_train_scaled, y_train)

best_svm_params = grid_svm.best_params_
print("Лучшие параметры Linear SVM:", best_svm_params)

best_svm = LinearSVC(**best_svm_params, random_state=42, class_weight='balanced', max_iter=10000)
svm_calibrated = CalibratedClassifierCV(best_svm, method='sigmoid', cv=3)
svm_calibrated.fit(X_train_scaled, y_train)
y_val_proba_svm_opt = svm_calibrated.predict_proba(X_val_scaled)[:, 1]

logloss_svm_opt = log_loss(y_val, y_val_proba_svm_opt)
roc_auc_svm_opt = roc_auc_score(y_val, y_val_proba_svm_opt)

prec_svm, rec_svm, thresh_svm = precision_recall_curve(y_val, y_val_proba_svm_opt)
f1_svm_curve = 2 * prec_svm * rec_svm / (prec_svm + rec_svm + 1e-9)
best_th_svm = thresh_svm[np.argmax(f1_svm_curve[:-1])]
y_val_pred_svm_opt = (y_val_proba_svm_opt >= best_th_svm).astype(int)

f1_svm_opt = f1_score(y_val, y_val_pred_svm_opt)
precision_svm_opt = precision_score(y_val, y_val_pred_svm_opt)
recall_svm_opt = recall_score(y_val, y_val_pred_svm_opt)

print("\nLinear SVM (оптимизированный):")
print(f"LogLoss: {logloss_svm_opt:.6f}")
print(f"ROC-AUC: {roc_auc_svm_opt:.6f}")
print(f"F1 (порог={best_th_svm:.3f}): {f1_svm_opt:.6f}")
print(f"Precision: {precision_svm_opt:.6f}")
print(f"Recall: {recall_svm_opt:.6f}\n")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Лучшие параметры Linear SVM: {'C': 0.01, 'loss': 'hinge', 'tol': 0.0001}

Linear SVM (оптимизированный):
LogLoss: 0.009888
ROC-AUC: 0.813291
F1 (порог=0.025): 0.096774
Precision: 0.060000
Recall: 0.250000



In [16]:
from xgboost import XGBClassifier

xgb_base = XGBClassifier(
    random_state=42,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    eval_metric='logloss'
)

param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(xgb_base, param_grid_xgb, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_xgb.fit(X_train_scaled, y_train)

best_xgb_params = grid_xgb.best_params_
print("Лучшие параметры XGBoost:", best_xgb_params)

best_xgb = XGBClassifier(**best_xgb_params, random_state=42,
                         scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
                         eval_metric='logloss')
best_xgb.fit(X_train_scaled, y_train)
y_val_proba_xgb_opt = best_xgb.predict_proba(X_val_scaled)[:, 1]

logloss_xgb_opt = log_loss(y_val, y_val_proba_xgb_opt)
roc_auc_xgb_opt = roc_auc_score(y_val, y_val_proba_xgb_opt)

prec_xgb, rec_xgb, thresh_xgb = precision_recall_curve(y_val, y_val_proba_xgb_opt)
f1_xgb_curve = 2 * prec_xgb * rec_xgb / (prec_xgb + rec_xgb + 1e-9)
best_th_xgb = thresh_xgb[np.argmax(f1_xgb_curve[:-1])]
y_val_pred_xgb_opt = (y_val_proba_xgb_opt >= best_th_xgb).astype(int)

f1_xgb_opt = f1_score(y_val, y_val_pred_xgb_opt)
precision_xgb_opt = precision_score(y_val, y_val_pred_xgb_opt)
recall_xgb_opt = recall_score(y_val, y_val_pred_xgb_opt)

print("\nXGBoost (оптимизированный):")
print(f"LogLoss: {logloss_xgb_opt:.6f}")
print(f"ROC-AUC: {roc_auc_xgb_opt:.6f}")
print(f"F1 (порог={best_th_xgb:.3f}): {f1_xgb_opt:.6f}")
print(f"Precision: {precision_xgb_opt:.6f}")
print(f"Recall: {recall_xgb_opt:.6f}\n")

Fitting 3 folds for each of 108 candidates, totalling 324 fits
Лучшие параметры XGBoost: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100, 'subsample': 0.8}

XGBoost (оптимизированный):
LogLoss: 0.172470
ROC-AUC: 0.748191
F1 (порог=0.944): 0.076923
Precision: 0.500000
Recall: 0.041667



In [17]:
import lightgbm as lgb

lgbm_base = lgb.LGBMClassifier(
    random_state=42,
    class_weight='balanced',
    verbose=-1
)

param_grid_lgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [4, 6, 8, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgb = GridSearchCV(lgbm_base, param_grid_lgb, cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)
grid_lgb.fit(X_train_scaled, y_train)

best_lgb_params = grid_lgb.best_params_
print("Лучшие параметры LightGBM:", best_lgb_params)

best_lgb = lgb.LGBMClassifier(**best_lgb_params, random_state=42, class_weight='balanced', verbose=-1)
best_lgb.fit(X_train_scaled, y_train)
y_val_proba_lgb_opt = best_lgb.predict_proba(X_val_scaled)[:, 1]

logloss_lgb_opt = log_loss(y_val, y_val_proba_lgb_opt)
roc_auc_lgb_opt = roc_auc_score(y_val, y_val_proba_lgb_opt)

prec_lgb, rec_lgb, thresh_lgb = precision_recall_curve(y_val, y_val_proba_lgb_opt)
f1_lgb_curve = 2 * prec_lgb * rec_lgb / (prec_lgb + rec_lgb + 1e-9)
best_th_lgb = thresh_lgb[np.argmax(f1_lgb_curve[:-1])]
y_val_pred_lgb_opt = (y_val_proba_lgb_opt >= best_th_lgb).astype(int)

f1_lgb_opt = f1_score(y_val, y_val_pred_lgb_opt)
precision_lgb_opt = precision_score(y_val, y_val_pred_lgb_opt)
recall_lgb_opt = recall_score(y_val, y_val_pred_lgb_opt)

print("\nLightGBM (оптимизированный):")
print(f"LogLoss: {logloss_lgb_opt:.6f}")
print(f"ROC-AUC: {roc_auc_lgb_opt:.6f}")
print(f"F1 (порог={best_th_lgb:.3f}): {f1_lgb_opt:.6f}")
print(f"Precision: {precision_lgb_opt:.6f}")
print(f"Recall: {recall_lgb_opt:.6f}\n")

Fitting 3 folds for each of 144 candidates, totalling 432 fits
Лучшие параметры LightGBM: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 50, 'subsample': 0.8}

LightGBM (оптимизированный):
LogLoss: 0.086460
ROC-AUC: 0.756175
F1 (порог=0.863): 0.088889
Precision: 0.095238
Recall: 0.083333



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Построим стэкинг-ансамбль из трёх базовых моделей – Linear SVM, XGBoost и LightGBM (каждая обучена с оптимальными параметрами). В качестве мета-классификатора используем логистическую регрессию. Эксперимент покажет, превзойдёт ли комбинация этих моделей лучшую одиночную модель (Linear SVM) по LogLoss и повысит ли стабильность предсказаний.

In [18]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

base_learners = [
    ('svm', svm_calibrated),
    ('xgb', best_xgb),
    ('lgb', best_lgb)
]

meta_learner = LogisticRegression(random_state=42)

stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=3,
    stack_method='predict_proba'
)

stacking_clf.fit(X_train_scaled, y_train)
y_val_proba_stack = stacking_clf.predict_proba(X_val_scaled)[:, 1]

logloss_stack = log_loss(y_val, y_val_proba_stack)
roc_auc_stack = roc_auc_score(y_val, y_val_proba_stack)

prec_stack, rec_stack, thresh_stack = precision_recall_curve(y_val, y_val_proba_stack)
f1_stack_curve = 2 * prec_stack * rec_stack / (prec_stack + rec_stack + 1e-9)
best_th_stack = thresh_stack[np.argmax(f1_stack_curve[:-1])]
y_val_pred_stack = (y_val_proba_stack >= best_th_stack).astype(int)

f1_stack = f1_score(y_val, y_val_pred_stack)
precision_stack = precision_score(y_val, y_val_pred_stack)
recall_stack = recall_score(y_val, y_val_pred_stack)

print("\nStacking")
print(f"LogLoss: {logloss_stack:.6f}")
print(f"ROC-AUC: {roc_auc_stack:.6f}")
print(f"F1 (порог={best_th_stack:.3f}): {f1_stack:.6f}")
print(f"Precision: {precision_stack:.6f}")
print(f"Recall: {recall_stack:.6f}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Stacking
LogLoss: 0.010777
ROC-AUC: 0.753072
F1 (порог=0.045): 0.074074
Precision: 0.333333
Recall: 0.041667


Применим SMOTE к обучающей выборке для двух ансамблевых моделей – XGBoost и LightGBM (без дополнительного взвешивания классов). Эксперимент проверит, даст ли синтетическое увеличение примеров класса «пульсар» преимущество по LogLoss и ROC‑AUC по сравнению с исходными моделями, которые использовали веса классов.

In [19]:
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from sklearn.metrics import log_loss, roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("Размер обучающей выборки после SMOTE:", X_train_smote.shape)
print("Распределение классов после SMOTE:\n", pd.Series(y_train_smote).value_counts())

xgb_smote = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_smote.fit(X_train_smote, y_train_smote)
y_val_proba_xgb_smote = xgb_smote.predict_proba(X_val_scaled)[:, 1]

logloss_xgb_smote = log_loss(y_val, y_val_proba_xgb_smote)
roc_auc_xgb_smote = roc_auc_score(y_val, y_val_proba_xgb_smote)

prec_xgb_s, rec_xgb_s, thresh_xgb_s = precision_recall_curve(y_val, y_val_proba_xgb_smote)
f1_xgb_s_curve = 2 * prec_xgb_s * rec_xgb_s / (prec_xgb_s + rec_xgb_s + 1e-9)
best_th_xgb_s = thresh_xgb_s[np.argmax(f1_xgb_s_curve[:-1])]
y_val_pred_xgb_s = (y_val_proba_xgb_smote >= best_th_xgb_s).astype(int)

f1_xgb_smote = f1_score(y_val, y_val_pred_xgb_s)
precision_xgb_smote = precision_score(y_val, y_val_pred_xgb_s)
recall_xgb_smote = recall_score(y_val, y_val_pred_xgb_s)

print("\nXGBoost + SMOTE")
print(f"LogLoss: {logloss_xgb_smote:.6f}")
print(f"ROC-AUC: {roc_auc_xgb_smote:.6f}")
print(f"F1 (оптим. порог={best_th_xgb_s:.3f}): {f1_xgb_smote:.6f}")
print(f"Precision: {precision_xgb_smote:.6f}")
print(f"Recall: {recall_xgb_smote:.6f}")

Размер обучающей выборки после SMOTE: (96444, 8)
Распределение классов после SMOTE:
 Class
0    48222
1    48222
Name: count, dtype: int64

XGBoost + SMOTE
LogLoss: 0.115070
ROC-AUC: 0.672118
F1 (оптим. порог=0.901): 0.038095
Precision: 0.024691
Recall: 0.083333


In [20]:
lgb_smote = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    verbose=-1
)
lgb_smote.fit(X_train_smote, y_train_smote)
y_val_proba_lgb_smote = lgb_smote.predict_proba(X_val_scaled)[:, 1]

logloss_lgb_smote = log_loss(y_val, y_val_proba_lgb_smote)
roc_auc_lgb_smote = roc_auc_score(y_val, y_val_proba_lgb_smote)

prec_lgb_s, rec_lgb_s, thresh_lgb_s = precision_recall_curve(y_val, y_val_proba_lgb_smote)
f1_lgb_s_curve = 2 * prec_lgb_s * rec_lgb_s / (prec_lgb_s + rec_lgb_s + 1e-9)
best_th_lgb_s = thresh_lgb_s[np.argmax(f1_lgb_s_curve[:-1])]
y_val_pred_lgb_s = (y_val_proba_lgb_smote >= best_th_lgb_s).astype(int)

f1_lgb_smote = f1_score(y_val, y_val_pred_lgb_s)
precision_lgb_smote = precision_score(y_val, y_val_pred_lgb_s)
recall_lgb_smote = recall_score(y_val, y_val_pred_lgb_s)

print("\nLightGBM + SMOTE")
print(f"LogLoss: {logloss_lgb_smote:.6f}")
print(f"ROC-AUC: {roc_auc_lgb_smote:.6f}")
print(f"F1 (оптим. порог={best_th_lgb_s:.3f}): {f1_lgb_smote:.6f}")
print(f"Precision: {precision_lgb_smote:.6f}")
print(f"Recall: {recall_lgb_smote:.6f}")


LightGBM + SMOTE
LogLoss: 0.114601
ROC-AUC: 0.672020
F1 (оптим. порог=0.918): 0.049383
Precision: 0.035088
Recall: 0.083333


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
results = {
    'Model': [
        'Logistic Regression', 'KNN (k=5)', 'Decision Tree',
        'Gaussian NB', 'Linear SVM',
        'Random Forest', 'XGBoost', 'LightGBM',
        'Linear SVM (opt)', 'XGBoost (opt)', 'LightGBM (opt)',
        'XGBoost + SMOTE', 'LightGBM + SMOTE',
        'Stacking (SVM+XGB+LGB)'
    ],
    'LogLoss': [
        logloss_val, logloss_knn, logloss_dt,
        logloss_nb, logloss_svm,
        logloss_rf, logloss_xgb, logloss_lgb,
        logloss_svm_opt, logloss_xgb_opt, logloss_lgb_opt,
        logloss_xgb_smote, logloss_lgb_smote,
        logloss_stack
    ],
    'ROC-AUC': [
        roc_auc_val, roc_auc_knn, roc_auc_dt,
        roc_auc_nb, roc_auc_svm,
        roc_auc_rf, roc_auc_xgb, roc_auc_lgb,
        roc_auc_svm_opt, roc_auc_xgb_opt, roc_auc_lgb_opt,
        roc_auc_xgb_smote, roc_auc_lgb_smote,
        roc_auc_stack
    ],
    'F1': [
        f1_val, f1_knn, f1_dt,
        f1_nb, f1_svm,
        f1_score_rf, f1_score_xgb, f1_score_lgb,
        f1_svm_opt, f1_xgb_opt, f1_lgb_opt,
        f1_xgb_smote, f1_lgb_smote,
        f1_stack
    ],
    'Precision': [
        precision_val, precision_knn_val, precision_dt_val,
        precision_nb, precision_svm,
        precision_rf, precision_xgb, precision_lgb,
        precision_svm_opt, precision_xgb_opt, precision_lgb_opt,
        precision_xgb_smote, precision_lgb_smote,
        precision_stack
    ],
    'Recall': [
        recall_val, recall_knn_val, recall_dt_val,
        recall_nb, recall_svm,
        recall_rf, recall_xgb, recall_lgb,
        recall_svm_opt, recall_xgb_opt, recall_lgb_opt,
        recall_xgb_smote, recall_lgb_smote,
        recall_stack
    ]
}

df_results = pd.DataFrame(results)
df_results = df_results.sort_values('LogLoss')

print("Сравнение всех моделей, включая эксперементальные версии")
print(df_results.to_string(index=False))

Сравнение всех моделей, включая эксперементальные версии
                 Model  LogLoss  ROC-AUC       F1  Precision   Recall
      Linear SVM (opt) 0.009888 0.813291 0.096774   0.060000 0.250000
            Linear SVM 0.009925 0.806066 0.100000   0.062500 0.250000
Stacking (SVM+XGB+LGB) 0.010777 0.753072 0.074074   0.333333 0.041667
               XGBoost 0.030202 0.686632 0.041667   0.041667 0.041667
              LightGBM 0.032365 0.651747 0.022727   0.015625 0.041667
           Gaussian NB 0.033321 0.780907 0.052632   0.033333 0.125000
             KNN (k=5) 0.048682 0.559479 0.048387   0.030000 0.125000
         Random Forest 0.056485 0.760259 0.037736   0.034483 0.041667
        LightGBM (opt) 0.086460 0.756175 0.088889   0.095238 0.083333
         Decision Tree 0.087322 0.499533 0.002977   0.001491 1.000000
      LightGBM + SMOTE 0.114601 0.672020 0.049383   0.035088 0.083333
       XGBoost + SMOTE 0.115070 0.672118 0.038095   0.024691 0.083333
         XGBoost (opt) 0.172470 0

Лучший результат по главной метрике LogLoss показал оптимизированный Linear SVM, незначительно опередив базовый Linear SVM, что подтверждает эффективность подобранных гиперпараметров. Стэкинг трёх моделей (SVM, XGBoost, LightGBM) продемонстрировал более высокий LogLoss и заметно более низкий ROC‑AUC по сравнению с лучшим одиночным SVM, поэтому от его использования решено отказаться. Оптимизация XGBoost и LightGBM привела к неожиданному ухудшению LogLoss - вероятно, из-за переобучения при подборе параметров. Применение SMOTE к XGBoost и LightGBM дало катастрофический рост LogLoss, что делает синтетическую балансировку непригодной для данной задачи - очевидно, добавление искусственных примеров нарушило калибровку вероятностей. Таким образом, финальной моделью выбирается оптимизированный Linear SVM как обеспечивающий наименьший LogLoss при высоком ROC‑AUC и сбалансированных Precision/Recall.

Финальной моделью для задачи бинарной классификации пульсаров выбран оптимизированный Linear SVM. Главное преимущество этой модели — минимальное значение LogLoss среди всех протестированных алгоритмов, что критически важно, поскольку именно эта метрика является целевой в соревновании Kaggle. Модель также демонстрирует высокий ROC-AUC (0,813), подтверждающий её способность разделять классы, и сбалансированные Precision/Recall (0,06 и 0,25) с учётом сильного дисбаланса (пульсары ≈0,15% выборки). Линейный SVM прост в интерпретации, устойчив к переобучению благодаря регуляризации, требует меньше вычислительных ресурсов по сравнению с ансамблями и легко калибрует вероятности после применения сигмоидной калибровки. Основной недостаток модели — предположение о линейной разделимости классов, однако высокий ROC-AUC свидетельствует о том, что линейная зависимость между признаками и целевой переменной достаточно сильна. Кроме того, модель может быть менее гибкой на более сложных данных, но в рамках текущего датасета это не является проблемой. Альтернативные подходы не только не улучшили LogLoss, но в большинстве случаев значительно его ухудшили. Таким образом, выбор оптимизированного Linear SVM полностью обоснован достигнутыми метриками и практическими соображениями.